# Module 2 — Fraud Detection: Anomaly Scoring + Cost-Sensitive Classification

**Business context:** Identify suspicious claims using a two-stage pipeline:
1. Isolation Forest + LOF generate anomaly scores as unsupervised signals.
2. CatBoost classifier with a cost matrix (FN cost = €5,000, FP cost = €150) catches fraud.

**Output:** Fraud risk score per claim with business impact calculation.

---
| Stage | Method | Role |
|---|---|---|
| Anomaly scoring | Isolation Forest + LOF | Unsupervised prior |
| Supervised model | CatBoost + cost matrix | Decision boundary |
| Threshold optimisation | Precision-recall cost curve | Minimise €-cost |
| Explainability | SHAP TreeExplainer | Feature attribution |
| Business impact | Savings vs. baseline | Quantified ROI |

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.catboost
import shap

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Project modules
from claims.data import load_fremtpl2, build_claims_dataset, claims_only
from claims.features import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    add_fraud_features,
)
from claims.fraud.anomaly import compute_anomaly_scores, create_fraud_proxy_labels
from claims.fraud.supervised import (
    cost_sensitive_catboost,
    optimal_threshold,
    plot_cost_curve,
    business_impact,
    FN_COST,
    FP_COST,
)
from claims.evaluation import classification_report_dict, gini_coefficient

print("All imports successful.")
print(f"Cost matrix — FN: €{FN_COST:,}  |  FP: €{FP_COST:,}")

## 2. Data Preparation

In [ ]:
# Load freMTPL2 from OpenML (cached after first download)
print("Loading freMTPL2 dataset from OpenML...")
freq, sev = load_fremtpl2()
print(f"Frequency table: {freq.shape}  |  Severity table: {sev.shape}")

# Join and engineer targets
df_all = build_claims_dataset(freq, sev)
print(f"Combined dataset: {df_all.shape}")

# Restrict to claim records (fraud analysis only applies to filed claims)
df_claims = claims_only(df_all)
print(f"Claims-only subset: {df_claims.shape}")

# Engineer fraud interaction features
df_claims = add_fraud_features(df_claims)
print("Fraud features added:", ["HighRiskDriver", "UrbanArea", "PenalisedDriver", "RiskInteraction"])

# Create proxy fraud labels from actuarial rules
df_claims = create_fraud_proxy_labels(df_claims, severity_col="AvgSeverity", percentile=97.5)
fraud_rate = df_claims["FraudProxy"].mean()
print(f"\nFraud proxy label rate: {fraud_rate:.2%}  ({df_claims['FraudProxy'].sum():,} flagged)")

df_claims.head(3)

## 3. Feature Matrix for Anomaly Detection

In [ ]:
# Anomaly detection uses only numeric features (IsolationForest/LOF require numeric input)
ANOMALY_FEATURES = NUMERIC_FEATURES + ["HighRiskDriver", "UrbanArea", "PenalisedDriver", "RiskInteraction"]

X_raw = df_claims[ANOMALY_FEATURES].copy()

# Impute missing values
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_raw)

# Scale features (important for LOF which uses distance-based neighbourhood)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features used: {ANOMALY_FEATURES}")

# Check for any residual NaNs
nan_count = np.isnan(X_scaled).sum()
print(f"NaN count after imputation: {nan_count}")

## 4. Isolation Forest

In [ ]:
print("Fitting Isolation Forest (n_estimators=200, contamination=0.05)...")
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1,
)
iso_forest.fit(X_scaled)

# decision_function: negative values are more anomalous; flip sign so higher = more anomalous
iso_scores = -iso_forest.decision_function(X_scaled)

print(f"IsoForest score range: [{iso_scores.min():.4f}, {iso_scores.max():.4f}]")
print(f"Flagged as outliers (score > 0): {(iso_forest.predict(X_scaled) == -1).sum():,}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(iso_scores, bins=80, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(np.percentile(iso_scores, 95), color="red", linestyle="--",
           label="95th percentile (potential fraud zone)")
ax.set_xlabel("Isolation Forest Anomaly Score (higher = more anomalous)")
ax.set_ylabel("Count")
ax.set_title("Isolation Forest Score Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Local Outlier Factor

In [ ]:
print("Fitting Local Outlier Factor (n_neighbors=20, contamination=0.05)...")
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    novelty=False,
    n_jobs=-1,
)
lof.fit(X_scaled)

# negative_outlier_factor_: more negative = more anomalous; flip sign
lof_scores = -lof.negative_outlier_factor_

print(f"LOF score range: [{lof_scores.min():.4f}, {lof_scores.max():.4f}]")
print(f"Flagged as outliers by LOF: {(lof_scores > np.percentile(lof_scores, 95)).sum():,}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lof_scores, bins=80, color="darkorange", edgecolor="white", alpha=0.85)
ax.axvline(np.percentile(lof_scores, 95), color="red", linestyle="--",
           label="95th percentile (potential fraud zone)")
ax.set_xlabel("LOF Anomaly Score (higher = more anomalous)")
ax.set_ylabel("Count")
ax.set_title("Local Outlier Factor Score Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Combined Anomaly Score

In [ ]:
# Use the module function which normalises and averages both detectors
print("Computing combined anomaly score via compute_anomaly_scores()...")
anomaly_scores = compute_anomaly_scores(
    X_scaled,
    contamination=0.05,
    n_estimators=200,
    n_neighbors=20,
    random_state=42,
)

# Attach to DataFrame as a new feature
df_claims["AnomalyScore"] = anomaly_scores

print(f"AnomalyScore range: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")
print(f"Mean anomaly score: {anomaly_scores.mean():.4f}")

# Top-10 most anomalous claims
print("\nTop-10 most anomalous claims:")
top10_cols = ["IDpol", "AvgSeverity", "BonusMalus", "DrivAge", "VehAge",
              "FraudProxy", "AnomalyScore"]
display_cols = [c for c in top10_cols if c in df_claims.columns]
top10 = df_claims.nlargest(10, "AnomalyScore")[display_cols]
print(top10.to_string(index=False))

## 7. Supervised CatBoost Fraud Model

In [ ]:
# Feature set: numeric + categorical (raw strings) + anomaly score
FRAUD_NUM_FEATURES = NUMERIC_FEATURES + ["HighRiskDriver", "UrbanArea",
                                         "PenalisedDriver", "RiskInteraction",
                                         "AnomalyScore"]
FRAUD_CAT_FEATURES = CATEGORICAL_FEATURES
ALL_FRAUD_FEATURES = FRAUD_NUM_FEATURES + FRAUD_CAT_FEATURES

# Class imbalance summary
fraud_count = df_claims["FraudProxy"].sum()
total_count = len(df_claims)
print(f"Class distribution — Fraud: {fraud_count:,} ({fraud_count/total_count:.2%})  "
      f"Legitimate: {total_count - fraud_count:,} ({1 - fraud_count/total_count:.2%})")
print(f"Imbalance ratio: 1 : {(total_count - fraud_count) / max(fraud_count, 1):.1f}")

# Build X (keep raw string categoricals for CatBoost) and y
X_fraud = df_claims[ALL_FRAUD_FEATURES].copy()
y_fraud = df_claims["FraudProxy"].values

# Convert numerics to float; leave categoricals as strings
for col in FRAUD_NUM_FEATURES:
    X_fraud[col] = pd.to_numeric(X_fraud[col], errors="coerce").fillna(
        X_fraud[col].median() if X_fraud[col].dtype != object else 0
    )

# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

# CatBoost expects cat_features as column-name strings when X is a DataFrame
cat_feature_names = FRAUD_CAT_FEATURES

# Build and fit cost-sensitive model
fraud_model = cost_sensitive_catboost(
    cat_features=cat_feature_names,
    fn_cost=FN_COST,
    fp_cost=FP_COST,
    iterations=500,
    learning_rate=0.05,
    depth=6,
    verbose=0,
)
print("\nFitting CatBoost fraud model...")
fraud_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=50,
)
print("Training complete.")

In [ ]:
# Predict probabilities on test set
y_prob = fraud_model.predict_proba(X_test)[:, 1]

# Default-threshold metrics
metrics = classification_report_dict(y_test, y_prob, threshold=0.5)
print("Test-set metrics (threshold=0.5):")
for k, v in metrics.items():
    print(f"  {k:<18}: {v:.4f}")

## 8. Threshold Optimisation

In [ ]:
# Find the threshold that minimises total expected cost (FN*€5000 + FP*€150)
opt_thresh = optimal_threshold(y_test, y_prob, fn_cost=FN_COST, fp_cost=FP_COST)
print(f"Optimal decision threshold: {opt_thresh:.4f}")

# Re-evaluate at optimal threshold
metrics_opt = classification_report_dict(y_test, y_prob, threshold=opt_thresh)
print("\nTest-set metrics at optimal threshold:")
for k, v in metrics_opt.items():
    print(f"  {k:<18}: {v:.4f}")

# Cost curve visualisation
fig, ax = plt.subplots(figsize=(10, 5))
plot_cost_curve(y_test, y_prob, fn_cost=FN_COST, fp_cost=FP_COST, ax=ax)
plt.tight_layout()
plt.show()

## 9. SHAP Analysis

In [ ]:
print("Computing SHAP values with TreeExplainer...")
explainer = shap.TreeExplainer(fraud_model)

# Use a sample for speed (SHAP on full test set can be slow)
shap_sample_size = min(500, len(X_test))
X_shap = X_test.iloc[:shap_sample_size]
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values computed for {shap_sample_size} samples.")
print(f"SHAP values shape: {np.array(shap_values).shape}")

In [ ]:
# Summary bar plot — mean absolute SHAP values per feature
print("Top fraud features by mean |SHAP|:")
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="bar",
    max_display=15,
    show=True,
)

In [ ]:
# Beeswarm plot — direction and magnitude of each feature's effect
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="dot",
    max_display=15,
    show=True,
)

## 10. Business Impact

In [ ]:
# Compute business impact at optimal threshold vs no-model baseline
impact_df = business_impact(
    y_test, y_prob, threshold=opt_thresh, fn_cost=FN_COST, fp_cost=FP_COST
)

print("Business Impact Summary")
print("=" * 50)
print(impact_df.to_string(index=False))
print("\nBaseline: flag nothing (all frauds go undetected).")
print(f"Cost matrix: FN = €{FN_COST:,}  |  FP = €{FP_COST:,}")

## 11. MLflow Logging

In [ ]:
mlflow.set_experiment("fraud-detection")

with mlflow.start_run(run_name="catboost-cost-sensitive"):
    # Log model hyperparameters
    mlflow.log_params({
        "model": "CatBoostClassifier",
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 6,
        "fn_cost": FN_COST,
        "fp_cost": FP_COST,
        "scale_pos_weight": FN_COST / FP_COST,
        "contamination": 0.05,
        "anomaly_n_estimators": 200,
        "lof_n_neighbors": 20,
    })

    # Log metrics at optimal threshold
    mlflow.log_metrics({
        "roc_auc": metrics_opt["roc_auc"],
        "avg_precision": metrics_opt["avg_precision"],
        "brier_score": metrics_opt["brier_score"],
        "cohen_kappa": metrics_opt["cohen_kappa"],
        "gini": metrics_opt["gini"],
        "optimal_threshold": opt_thresh,
        "fraud_rate": fraud_rate,
        "shap_sample_size": shap_sample_size,
    })

    # Log the CatBoost model
    mlflow.catboost.log_model(fraud_model, artifact_path="fraud_model")

    run_id = mlflow.active_run().info.run_id

print(f"MLflow run logged: {run_id}")
print("Experiment: fraud-detection")
print(f"  roc_auc          : {metrics_opt['roc_auc']:.4f}")
print(f"  gini             : {metrics_opt['gini']:.4f}")
print(f"  optimal_threshold: {opt_thresh:.4f}")